# DCASE 2025 Task 2 — Strict Train-Only Feature Selection

Machine-specific subsets are selected using only 990 source-normal + 10 target-normal training recordings. The final output is a copy-paste-ready `FEATURES_BY_MACHINE` dictionary.


In [ ]:
# -*- coding: utf-8 -*-
"""
DCASE 2025 Task 2
FROZEN-68 -> STRICT TRAIN-ONLY MACHINE-SPECIFIC FEATURE-SUBSET SELECTION
DUAL RAW kNN: SOURCE k=10, TARGET k=3, FINAL=min(dS,dT)
=======================================================================

For each unseen/evaluation machine, feature selection uses ONLY the
990 source-normal + 10 target-normal training recordings. Evaluation/test
recordings are not used by the subset-selection objective.

The fixed development-derived pool of 68 TDA descriptors is scaled with
parameters fitted on the pooled 1000 normal training recordings. Source and
target normal banks remain separate. Candidate subsets are ranked by a
normal-only reliability criterion based on source/target compactness, normal-tail
stability, target-normal acceptance by the source-normal geometry, source/target
stability balance, and a redundancy penalty.

SEARCH
------
D=1      : exhaustive 68 singles
D=2      : exhaustive C(68,2)=2278 pairs
D=3..20  : beam search + random injections

Candidates are calibrated within dimensionality using a robust z score.
The rank-1 subset for each machine is printed at the end as an evaluator-ready
FEATURES_BY_MACHINE Python dictionary.
"""

from __future__ import annotations

import glob
import json
import math
import os
import random
import shutil
import time
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree



# =============================================================================
# 0. CONFIG -- CHANGE THESE IF YOU WANT A DEEPER / FASTER SEARCH
# =============================================================================

MACHINE_TYPES = [
    "AutoTrash",
    "BandSealer",
    "CoffeeGrinder",
    "HomeCamera",
    "Polisher",
    "ScrewFeeder",
    "ToyPet",
    "ToyRCCar",
]

BEST_FEATURE_POOL_68 = [
    "H0_landscape_auc",
    "H0_landscape_l1",
    "H0_landscape_l2",
    "H0_landscape_layer5_auc",
    "H0_lifetime_skew",
    "H0_pimage_entropy",
    "H0_pimage_min",
    "H1_betti_max",
    "H1_birth_iqr",
    "H1_birth_var",
    "H1_death_iqr",
    "H1_death_max",
    "H1_death_min",
    "H1_death_var",
    "H1_min_lifetime",
    "H1_min_midlife",
    "H1_q25_lifetime",
    "H1_to_H0_num_points_ratio",
    "H1_weighted_midlife_std",
    "H0_betti_max",
    "H0_betti_num_peaks",
    "H0_landscape_layer2_max",
    "H0_landscape_layer3_max",
    "H0_mean_birth_death_ratio",
    "H0_range_lifetime",
    "H0_top3_share",
    "H1_landscape_layer2_auc",
    "H1_landscape_layer2_mean",
    "H1_landscape_layer3_mean",
    "H1_landscape_layer4_auc",
    "H1_max_lifetime",
    "H1_minus_H0_entropy",
    "H1_silhouette_std",
    "H1_std_birth_death_ratio",
    "H0_birth_skew",
    "H0_death_skew",
    "H0_persistence_entropy",
    "H0_top1_share",
    "H1_normalized_persistence_entropy",
    "H1_num_points",
    "H1_to_H0_entropy_ratio",
    "H0_death_min",
    "H0_landscape_entropy",
    "H1_death_std",
    "H1_landscape_layer2_max",
    "H1_pimage_min",
    "H0_betti_l1",
    "H0_birth_max",
    "H0_death_q25",
    "H0_landscape_layer5_max",
    "H0_max_midlife",
    "H0_median_lifetime",
    "H0_num_points",
    "H0_pimage_energy",
    "H1_birth_max",
    "H1_silhouette_entropy",
    "H1_tail_share_q90",
    "H1_tail_share_q95",
    "H0_landscape_layer1_mean",
    "H1_landscape_layer1_auc",
    "H1_to_H0_max_lifetime_ratio",
    "H0_birth_kurtosis",
    "H0_landscape_layer4_max",
    "H0_q75_lifetime",
    "H1_betti_l2",
    "H1_betti_std",
    "H1_mean_birth_death_ratio",
    "H1_persistence_entropy",
]

N_FEATURES = len(BEST_FEATURE_POOL_68)
PAIR_COUNT = math.comb(N_FEATURES, 2)

# Reference neighborhoods requested by the user.
SOURCE_K = 10
TARGET_K = 3

# Feature-subset dimensions.
MAX_DIM = 20

# Number of highest-ranked label-free subsets to report per machine.
TOP_N_SUBSETS = 10

# Scaling: fit on pooled 990+10 normal train only.
SCALER_MODE = "minmax_train"

# Divide Euclidean distances by sqrt(D) for numerical comparability across D.
DISTANCE_DIM_NORMALIZE = True

# Search depth.
BEAM_WIDTH = 24
EXPANSION_POOL_SIZE = 44
TOP_SINGLES_FOR_POOL = 30
TOP_PAIRS_FOR_POOL = 200
RANDOM_INJECTIONS_PER_DIM = 80
CALIBRATION_RANDOM_PER_DIM = 60
TOP_KEEP_PER_DIM = 80

RANDOM_SEED = 20260819

# Strict train-only normal-stability objective.
# Positive components are approximately bounded in [0,1].
W_SOURCE_COMPACTNESS = 0.20
W_TARGET_COMPACTNESS = 0.20
W_SOURCE_TAIL_STABILITY = 0.15
W_TARGET_TAIL_STABILITY = 0.15
W_TARGET_ACCEPTANCE = 0.15
W_DOMAIN_STABILITY_BALANCE = 0.15
W_REDUNDANCY_PENALTY = 0.10

SOURCE_ACCEPTANCE_QUANTILE = 0.99
SOURCE_TAIL_QUANTILE = 0.95
TARGET_TAIL_QUANTILE = 0.90

# Numerical safety.
EPS = 1e-12
MAX_Z = 50.0


# Output / Colab behavior.
AUTO_DOWNLOAD_ZIP = True

# Console output controls.
# SHOW_PROGRESS keeps only machine-level progress; VERBOSE enables deep search logs.
SHOW_PROGRESS = True
VERBOSE = False
STRICT_DCASE_COUNTS = True

# Auto-detect default input root.
if os.path.isdir("/content"):
    INPUT_DIR = "/content"
    OUTPUT_DIR = "/content/dcase2025_TRAIN_ONLY_68_feature_selection_reports"
else:
    INPUT_DIR = "/mnt/data"
    OUTPUT_DIR = "/mnt/data/dcase2025_TRAIN_ONLY_68_feature_selection_reports"

PER_MACHINE_DIR = os.path.join(OUTPUT_DIR, "per_machine")


# =============================================================================
# 1. BASIC UTILITIES
# =============================================================================

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def progress(message: str) -> None:
    if SHOW_PROGRESS:
        print(message)


def log(*args, **kwargs) -> None:
    if VERBOSE:
        print(*args, **kwargs)


def validate_config() -> None:
    if len(set(BEST_FEATURE_POOL_68)) != N_FEATURES:
        raise ValueError("BEST_FEATURE_POOL_68 contains duplicate feature names")
    if N_FEATURES != 68:
        raise ValueError(f"Frozen feature pool must contain 68 features; got {N_FEATURES}")
    if not (1 <= MAX_DIM <= N_FEATURES):
        raise ValueError(f"MAX_DIM must be in [1, {N_FEATURES}]; got {MAX_DIM}")
    if SOURCE_K < 1 or TARGET_K < 1:
        raise ValueError("SOURCE_K and TARGET_K must be positive")
    if not (0.0 < SOURCE_ACCEPTANCE_QUANTILE < 1.0):
        raise ValueError("SOURCE_ACCEPTANCE_QUANTILE must be in (0,1)")


def clean_output_dir() -> None:
    if os.path.isdir(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    ensure_dir(OUTPUT_DIR)
    ensure_dir(PER_MACHINE_DIR)


def normalize_file_id(value: object) -> str:
    s = str(value).strip().replace("\\", "/")
    return os.path.basename(s)


def infer_train_domain(series: pd.Series) -> np.ndarray:
    s = series.astype(str).str.lower()
    out = np.full(len(s), "", dtype=object)
    out[s.str.contains(r"(?:^|_)source(?:_|$)", regex=True)] = "source"
    out[s.str.contains(r"(?:^|_)target(?:_|$)", regex=True)] = "target"
    return out


def robust_scale_1d(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return 1.0

    q25, q75 = np.quantile(x, [0.25, 0.75])
    s = float(q75 - q25)
    if s > EPS:
        return s

    q10, q90 = np.quantile(x, [0.10, 0.90])
    s = float(q90 - q10)
    if s > EPS:
        return s

    s = float(np.std(x))
    if s > EPS:
        return s

    med_abs = float(np.median(np.abs(x))) if len(x) else 1.0
    return max(0.05 * max(med_abs, 1.0), EPS)


def robust_z(value: float, baseline_values: Sequence[float]) -> float:
    x = np.asarray(list(baseline_values), dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return 0.0
    med = float(np.median(x))
    scale = robust_scale_1d(x)
    z = (float(value) - med) / max(scale, EPS)
    return float(np.clip(z, -MAX_Z, MAX_Z))


def safe_log_positive(x: float) -> float:
    return float(np.log1p(max(0.0, float(x))))


def set_global_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def unique_tuples(items: Iterable[Tuple[int, ...]]) -> List[Tuple[int, ...]]:
    seen = set()
    out = []
    for item in items:
        t = tuple(sorted(int(v) for v in item))
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out


# =============================================================================
# 2. TRAIN FILE DISCOVERY -- TEST FILES ARE NOT USED FOR SELECTION
# =============================================================================

def find_machine_train_file(machine: str) -> str:
    """Find newest *_thr*.xlsx normal-train file for one machine."""
    all_xlsx = glob.glob(os.path.join(INPUT_DIR, "*.xlsx"))
    mk = machine.lower()
    matches = [
        p for p in all_xlsx
        if mk in os.path.basename(p).lower()
        and "_thr" in os.path.basename(p).lower()
        and not os.path.basename(p).startswith("~$")
    ]
    matches = sorted(matches, key=lambda p: (os.path.getmtime(p), p), reverse=True)
    if not matches:
        raise FileNotFoundError(
            f"{machine}: no *_thr*.xlsx normal-train file found under {INPUT_DIR}"
        )
    return matches[0]

# =============================================================================
# 3. DATA PREPARATION
# =============================================================================

@dataclass
class PreparedMachine:
    machine: str
    feature_names: List[str]
    train_path: str
    test_path: str

    train_file_ids: np.ndarray
    test_file_ids: np.ndarray
    train_domains: np.ndarray

    X_train_raw: np.ndarray
    X_test_raw: np.ndarray
    X_train_scaled: np.ndarray
    X_test_scaled: np.ndarray
    X_source: np.ndarray
    X_target: np.ndarray

    source_train_indices: np.ndarray
    target_train_indices: np.ndarray

    impute_median: np.ndarray
    scale_min: np.ndarray
    scale_max: np.ndarray
    scale_range: np.ndarray


def numeric_feature_frame(df: pd.DataFrame, features: Sequence[str]) -> np.ndarray:
    arr = (
        df[list(features)]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=np.float64)
    )
    arr[~np.isfinite(arr)] = np.nan
    return arr


def prepare_machine(machine: str) -> PreparedMachine:
    train_path = find_machine_train_file(machine)
    log("\n" + "=" * 120)
    log(f"PREPARING {machine} -- TRAIN ONLY")
    log("train:", os.path.basename(train_path))

    train_df = pd.read_excel(train_path)
    missing_train = [f for f in BEST_FEATURE_POOL_68 if f not in train_df.columns]
    if missing_train:
        raise ValueError(f"{machine}: frozen-68 missing from train: {missing_train}")

    if "file_id" in train_df.columns:
        train_ids = train_df["file_id"].map(normalize_file_id).to_numpy(dtype=str)
    elif "file_path" in train_df.columns:
        train_ids = train_df["file_path"].map(normalize_file_id).to_numpy(dtype=str)
    else:
        raise ValueError(f"{machine}: train needs file_id or file_path")

    domain_candidates = []
    if "file_id" in train_df.columns:
        domain_candidates.append(train_df["file_id"])
    if "file_path" in train_df.columns:
        domain_candidates.append(train_df["file_path"])

    train_domains = np.full(len(train_df), "", dtype=object)
    for series in domain_candidates:
        inferred = infer_train_domain(series)
        fill = train_domains == ""
        train_domains[fill] = inferred[fill]

    if np.any(train_domains == ""):
        bad = np.where(train_domains == "")[0][:10].tolist()
        raise ValueError(f"{machine}: cannot infer train domain at rows {bad}")

    src_mask = train_domains == "source"
    tgt_mask = train_domains == "target"
    n_src = int(src_mask.sum())
    n_tgt = int(tgt_mask.sum())

    if n_src != 990 or n_tgt != 10:
        message = (
            f"{machine}: expected source-normal=990, target-normal=10; "
            f"got source={n_src}, target={n_tgt}."
        )
        if STRICT_DCASE_COUNTS:
            raise ValueError(message)
        log(message)

    if n_src <= SOURCE_K or n_tgt <= TARGET_K:
        raise ValueError(
            f"{machine}: insufficient normal-bank size for kNN; "
            f"source={n_src}, target={n_tgt}, kS={SOURCE_K}, kT={TARGET_K}"
        )

    X_train_raw = numeric_feature_frame(train_df, BEST_FEATURE_POOL_68)
    finite_counts = np.isfinite(X_train_raw).sum(axis=0)
    if np.any(finite_counts == 0):
        bad_names = [BEST_FEATURE_POOL_68[i] for i in np.where(finite_counts == 0)[0]]
        raise ValueError(f"{machine}: no finite train values for features {bad_names}")

    impute_median = np.nanmedian(X_train_raw, axis=0)
    X_train_imp = np.asarray(X_train_raw, dtype=float).copy()
    bad = ~np.isfinite(X_train_imp)
    if np.any(bad):
        rr, cc = np.where(bad)
        X_train_imp[rr, cc] = impute_median[cc]

    if SCALER_MODE != "minmax_train":
        raise ValueError(f"Unsupported SCALER_MODE={SCALER_MODE}")

    scale_min = np.min(X_train_imp, axis=0)
    scale_max = np.max(X_train_imp, axis=0)
    scale_range = scale_max - scale_min
    bad_range = (~np.isfinite(scale_range)) | (scale_range <= EPS)
    if np.any(bad_range):
        bad_names = [BEST_FEATURE_POOL_68[i] for i in np.where(bad_range)[0]]
        raise ValueError(
            f"{machine}: constant/invalid frozen features under pooled-train MinMax: {bad_names}"
        )

    X_train_scaled = (X_train_imp - scale_min) / scale_range
    src_idx = np.where(src_mask)[0]
    tgt_idx = np.where(tgt_mask)[0]

    # Test placeholders are deliberately empty: test recordings are not loaded.
    empty_1d = np.asarray([], dtype=str)
    empty_2d = np.empty((0, N_FEATURES), dtype=float)

    return PreparedMachine(
        machine=machine,
        feature_names=list(BEST_FEATURE_POOL_68),
        train_path=train_path,
        test_path="",
        train_file_ids=train_ids,
        test_file_ids=empty_1d,
        train_domains=train_domains,
        X_train_raw=X_train_raw,
        X_test_raw=empty_2d,
        X_train_scaled=X_train_scaled,
        X_test_scaled=empty_2d,
        X_source=X_train_scaled[src_idx],
        X_target=X_train_scaled[tgt_idx],
        source_train_indices=src_idx,
        target_train_indices=tgt_idx,
        impute_median=impute_median,
        scale_min=scale_min,
        scale_max=scale_max,
        scale_range=scale_range,
    )


# =============================================================================
# 4. DUAL RAW kNN SCORING
# =============================================================================

def _query_mean(tree: cKDTree, X: np.ndarray, k: int) -> np.ndarray:
    k_eff = max(1, min(int(k), int(tree.n)))
    d, _ = tree.query(X, k=k_eff, workers=-1)
    d = np.asarray(d, dtype=float)
    if d.ndim == 1:
        d = d[:, None]
    return np.mean(d, axis=1)


def _loo_mean(tree: cKDTree, X: np.ndarray, k: int) -> np.ndarray:
    n = len(X)
    if n < 2:
        raise ValueError("LOO kNN needs at least 2 samples")
    k_eff = min(int(k), n - 1)
    d, _ = tree.query(X, k=k_eff + 1, workers=-1)
    d = np.asarray(d, dtype=float)
    if d.ndim == 1:
        d = d[:, None]
    # First neighbor is self at distance 0.
    return np.mean(d[:, 1:k_eff + 1], axis=1)


@dataclass
class ScoreBundle:
    subset: Tuple[int, ...]
    dim: int

    source_train_dS: np.ndarray
    source_train_dT: np.ndarray
    source_train_score: np.ndarray

    target_train_dS: np.ndarray
    target_train_dT: np.ndarray
    target_train_score: np.ndarray

    test_dS: np.ndarray
    test_dT: np.ndarray
    test_score: np.ndarray

    all_scores: np.ndarray
    all_is_train: np.ndarray
    all_is_test: np.ndarray
    all_origin: np.ndarray
    all_file_ids: np.ndarray


def compute_score_bundle(data: PreparedMachine, subset: Tuple[int, ...]) -> ScoreBundle:
    subset = tuple(sorted(int(j) for j in subset))
    idx = np.asarray(subset, dtype=int)
    dim = len(subset)
    if dim < 1:
        raise ValueError("empty subset")

    Xs = np.ascontiguousarray(data.X_source[:, idx], dtype=np.float64)
    Xt = np.ascontiguousarray(data.X_target[:, idx], dtype=np.float64)
    src_tree = cKDTree(Xs)
    tgt_tree = cKDTree(Xt)

    src_dS = _loo_mean(src_tree, Xs, SOURCE_K)
    src_dT = _query_mean(tgt_tree, Xs, TARGET_K)
    tgt_dS = _query_mean(src_tree, Xt, SOURCE_K)
    tgt_dT = _loo_mean(tgt_tree, Xt, TARGET_K)

    if DISTANCE_DIM_NORMALIZE:
        denom = math.sqrt(float(dim))
        src_dS = src_dS / denom
        src_dT = src_dT / denom
        tgt_dS = tgt_dS / denom
        tgt_dT = tgt_dT / denom

    src_score = np.minimum(src_dS, src_dT)
    tgt_score = np.minimum(tgt_dS, tgt_dT)
    train_scores = np.concatenate([src_score, tgt_score])
    n_train = len(train_scores)
    src_ids = data.train_file_ids[data.source_train_indices]
    tgt_ids = data.train_file_ids[data.target_train_indices]

    return ScoreBundle(
        subset=subset,
        dim=dim,
        source_train_dS=src_dS,
        source_train_dT=src_dT,
        source_train_score=src_score,
        target_train_dS=tgt_dS,
        target_train_dT=tgt_dT,
        target_train_score=tgt_score,
        test_dS=np.asarray([], dtype=float),
        test_dT=np.asarray([], dtype=float),
        test_score=np.asarray([], dtype=float),
        all_scores=train_scores,
        all_is_train=np.ones(n_train, dtype=bool),
        all_is_test=np.zeros(n_train, dtype=bool),
        all_origin=np.asarray(
            ["source_train"] * len(src_score) + ["target_train"] * len(tgt_score),
            dtype=object,
        ),
        all_file_ids=np.concatenate([src_ids, tgt_ids]),
    )


# =============================================================================
# 5. STRICT TRAIN-ONLY NORMAL-STABILITY OBJECTIVE
# =============================================================================

def mean_abs_redundancy(data: PreparedMachine, subset: Tuple[int, ...]) -> float:
    if len(subset) <= 1:
        return 0.0
    X = np.asarray(data.X_train_scaled[:, list(subset)], dtype=float)
    corr = np.corrcoef(X, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    iu = np.triu_indices(len(subset), k=1)
    vals = np.abs(corr[iu])
    return float(np.mean(vals)) if len(vals) else 0.0


def evaluate_normal_only(data: PreparedMachine, bundle: ScoreBundle) -> Dict[str, float]:
    src = np.asarray(bundle.source_train_dS, dtype=float)
    tgt = np.asarray(bundle.target_train_dT, dtype=float)
    tgt_to_src = np.asarray(bundle.target_train_dS, dtype=float)

    src_med = float(np.median(src))
    tgt_med = float(np.median(tgt))
    src_q95 = float(np.quantile(src, SOURCE_TAIL_QUANTILE))
    tgt_q90 = float(np.quantile(tgt, TARGET_TAIL_QUANTILE))
    src_q99 = float(np.quantile(src, SOURCE_ACCEPTANCE_QUANTILE))

    source_compactness = 1.0 / (1.0 + max(src_med, 0.0))
    target_compactness = 1.0 / (1.0 + max(tgt_med, 0.0))
    source_tail_stability = float(np.clip(src_med / max(src_q95, EPS), 0.0, 1.0))
    target_tail_stability = float(np.clip(tgt_med / max(tgt_q90, EPS), 0.0, 1.0))
    target_acceptance = float(np.mean(tgt_to_src <= src_q99))
    domain_stability_balance = float(
        np.clip(1.0 - abs(source_tail_stability - target_tail_stability), 0.0, 1.0)
    )
    redundancy_penalty = mean_abs_redundancy(data, bundle.subset)

    fitness = (
        W_SOURCE_COMPACTNESS * source_compactness
        + W_TARGET_COMPACTNESS * target_compactness
        + W_SOURCE_TAIL_STABILITY * source_tail_stability
        + W_TARGET_TAIL_STABILITY * target_tail_stability
        + W_TARGET_ACCEPTANCE * target_acceptance
        + W_DOMAIN_STABILITY_BALANCE * domain_stability_balance
        - W_REDUNDANCY_PENALTY * redundancy_penalty
    )

    return {
        "fitness": float(fitness),
        "source_compactness": float(source_compactness),
        "target_compactness": float(target_compactness),
        "source_tail_stability": float(source_tail_stability),
        "target_tail_stability": float(target_tail_stability),
        "target_acceptance": float(target_acceptance),
        "domain_stability_balance": float(domain_stability_balance),
        "redundancy_penalty": float(redundancy_penalty),
        "source_median_loo": float(src_med),
        "target_median_loo": float(tgt_med),
        "source_q95_loo": float(src_q95),
        "target_q90_loo": float(tgt_q90),
        "source_q99_accept_threshold": float(src_q99),
        "target_to_source_median": float(np.median(tgt_to_src)),
    }


# =============================================================================
# 6. SUBSET ENGINE WITH SCALAR CACHE
# =============================================================================

class SubsetEngine:
    def __init__(self, data: PreparedMachine):
        self.data = data
        self.cache: Dict[Tuple[int, ...], Dict[str, object]] = {}

    def evaluate(self, subset: Tuple[int, ...]) -> Dict[str, object]:
        subset = tuple(sorted(int(j) for j in subset))
        if subset in self.cache:
            return self.cache[subset]
        bundle = compute_score_bundle(self.data, subset)
        metrics = evaluate_normal_only(self.data, bundle)
        result: Dict[str, object] = {
            "subset": subset,
            "dim": len(subset),
            **metrics,
        }
        self.cache[subset] = result
        return result


def within_dim_ranking_key(r: Dict[str, object]) -> Tuple:
    return (
        float(r["fitness"]),
        float(r["target_acceptance"]),
        float(r["domain_stability_balance"]),
        float(r["target_tail_stability"]),
        float(r["source_tail_stability"]),
        -float(r["redundancy_penalty"]),
    )


def final_cross_dim_key(r: Dict[str, object]) -> Tuple:
    return (
        float(r.get("dim_z", -999.0)),
        float(r["fitness"]),
        float(r["target_acceptance"]),
        float(r["domain_stability_balance"]),
        -float(r["redundancy_penalty"]),
        -int(r["dim"]),
    )


def result_to_row(machine: str, r: Dict[str, object], rank: Optional[int] = None) -> Dict[str, object]:
    subset = tuple(r["subset"])
    return {
        "machine": machine,
        "rank": rank,
        "dim": int(r["dim"]),
        "dim_z": float(r.get("dim_z", np.nan)),
        "normal_only_fitness": float(r["fitness"]),
        "source_compactness": float(r["source_compactness"]),
        "target_compactness": float(r["target_compactness"]),
        "source_tail_stability": float(r["source_tail_stability"]),
        "target_tail_stability": float(r["target_tail_stability"]),
        "target_acceptance": float(r["target_acceptance"]),
        "domain_stability_balance": float(r["domain_stability_balance"]),
        "redundancy_penalty": float(r["redundancy_penalty"]),
        "source_median_loo": float(r["source_median_loo"]),
        "target_median_loo": float(r["target_median_loo"]),
        "source_q95_loo": float(r["source_q95_loo"]),
        "target_q90_loo": float(r["target_q90_loo"]),
        "features": ";".join(BEST_FEATURE_POOL_68[j] for j in subset),
        "feature_indices": ";".join(str(j) for j in subset),
    }

# =============================================================================
# 7. DIMENSION-CALIBRATED DEEP SEARCH D=1..20
# =============================================================================

def random_subset(dim: int, rng: np.random.Generator) -> Tuple[int, ...]:
    idx = rng.choice(len(BEST_FEATURE_POOL_68), size=dim, replace=False)
    return tuple(sorted(int(v) for v in idx))


def build_expansion_pool(
    singles: Sequence[Dict[str, object]],
    pairs: Sequence[Dict[str, object]],
) -> List[int]:
    votes = {j: 0.0 for j in range(len(BEST_FEATURE_POOL_68))}

    for rank, r in enumerate(singles[:TOP_SINGLES_FOR_POOL]):
        weight = TOP_SINGLES_FOR_POOL - rank
        for j in r["subset"]:
            votes[int(j)] += 5.0 * weight

    for rank, r in enumerate(pairs[:TOP_PAIRS_FOR_POOL]):
        weight = TOP_PAIRS_FOR_POOL - rank
        for j in r["subset"]:
            votes[int(j)] += weight

    ranked = sorted(votes, key=lambda j: (-votes[j], j))
    return ranked[:min(EXPANSION_POOL_SIZE, len(ranked))]


def dimension_baseline(
    engine: SubsetEngine,
    dim: int,
    known_results: Sequence[Dict[str, object]],
    rng: np.random.Generator,
) -> List[float]:
    if dim in (1, 2):
        return [float(r["fitness"]) for r in known_results]

    vals = []
    seen = set()
    attempts = 0
    max_attempts = CALIBRATION_RANDOM_PER_DIM * 20
    while len(vals) < CALIBRATION_RANDOM_PER_DIM and attempts < max_attempts:
        attempts += 1
        s = random_subset(dim, rng)
        if s in seen:
            continue
        seen.add(s)
        vals.append(float(engine.evaluate(s)["fitness"]))

    if len(vals) < 10:
        # Fallback to the dimension's explored results.
        vals.extend(float(r["fitness"]) for r in known_results)
    return vals


def attach_dimension_z(
    results: Sequence[Dict[str, object]],
    baseline: Sequence[float],
) -> None:
    for r in results:
        r["dim_z"] = robust_z(float(r["fitness"]), baseline)


def search_one_machine(data: PreparedMachine) -> Dict[str, object]:
    t0 = time.time()
    machine = data.machine
    out_dir = os.path.join(PER_MACHINE_DIR, machine)
    ensure_dir(out_dir)

    progress(f"Searching {machine}...")
    log("\n" + "#" * 120)
    log(f"SEARCHING {machine}: STRICT TRAIN-ONLY NORMAL-STABILITY SEARCH")
    log("#" * 120)

    engine = SubsetEngine(data)
    rng = np.random.default_rng(RANDOM_SEED + sum(ord(c) for c in machine))

    dim_results: Dict[int, List[Dict[str, object]]] = {}
    best_by_dim: List[Dict[str, object]] = []

    # -------------------------------------------------------------------------
    # D=1 exhaustive
    # -------------------------------------------------------------------------
    log(f"\n[D=1] exhaustive {N_FEATURES} singles")
    singles = [engine.evaluate((j,)) for j in range(N_FEATURES)]
    singles.sort(key=within_dim_ranking_key, reverse=True)
    dim_results[1] = singles
    base1 = dimension_baseline(engine, 1, singles, rng)
    attach_dimension_z(singles, base1)
    best_by_dim.append(singles[0])
    log(f"best D=1 fitness={singles[0]['fitness']:.4f} z={singles[0]['dim_z']:.3f}")

    # -------------------------------------------------------------------------
    # D=2 exhaustive
    # -------------------------------------------------------------------------
    log(f"\n[D=2] exhaustive {PAIR_COUNT} pairs")
    pairs: List[Dict[str, object]] = []
    count = 0
    for a in range(N_FEATURES):
        for b in range(a + 1, N_FEATURES):
            pairs.append(engine.evaluate((a, b)))
            count += 1
            if count % 400 == 0:
                log(f"  pairs {count}/{PAIR_COUNT}")

    pairs.sort(key=within_dim_ranking_key, reverse=True)
    dim_results[2] = pairs
    base2 = dimension_baseline(engine, 2, pairs, rng)
    attach_dimension_z(pairs, base2)
    best_by_dim.append(pairs[0])
    log(f"best D=2 fitness={pairs[0]['fitness']:.4f} z={pairs[0]['dim_z']:.3f}")

    expansion_pool = build_expansion_pool(singles, pairs)
    log("\nExpansion pool:")
    for j in expansion_pool:
        log(f"  {j:02d} {BEST_FEATURE_POOL_68[j]}")

    # Beam starts from top D=2 candidates.
    beam = pairs[:BEAM_WIDTH]

    # -------------------------------------------------------------------------
    # D=3..MAX_DIM beam + random injection
    # -------------------------------------------------------------------------
    for dim in range(3, MAX_DIM + 1):
        log(f"\n[D={dim}] beam search")

        candidate_subsets: List[Tuple[int, ...]] = []
        for parent in beam:
            parent_set = set(int(v) for v in parent["subset"])
            for j in expansion_pool:
                if j not in parent_set:
                    candidate_subsets.append(tuple(sorted((*parent["subset"], j))))

        # Random injections reduce the chance that forward beam gets trapped.
        for _ in range(RANDOM_INJECTIONS_PER_DIM):
            candidate_subsets.append(random_subset(dim, rng))

        candidate_subsets = unique_tuples(candidate_subsets)
        log(f"  evaluating {len(candidate_subsets)} unique candidates")

        current = []
        for i, subset in enumerate(candidate_subsets, start=1):
            current.append(engine.evaluate(subset))
            if i % 250 == 0:
                log(f"    {i}/{len(candidate_subsets)}")

        current.sort(key=within_dim_ranking_key, reverse=True)
        # Keep a generous diagnostic slice; beam itself remains BEAM_WIDTH.
        dim_results[dim] = current[:max(TOP_KEEP_PER_DIM, BEAM_WIDTH)]

        baseline = dimension_baseline(engine, dim, current, rng)
        attach_dimension_z(dim_results[dim], baseline)
        # Ensure best candidate z is attached even if object references differ.
        attach_dimension_z(current[:1], baseline)
        best_by_dim.append(current[0])

        log(f"best D={dim} fitness={current[0]['fitness']:.4f} z={current[0]['dim_z']:.3f}")

        beam = current[:BEAM_WIDTH]

    # -------------------------------------------------------------------------
    # Save best by dimension.
    # -------------------------------------------------------------------------
    best_dim_rows = []
    for dim in range(1, MAX_DIM + 1):
        r = max(dim_results[dim], key=within_dim_ranking_key)
        best_dim_rows.append(result_to_row(machine, r))

    best_dim_df = pd.DataFrame(best_dim_rows)
    best_dim_df.to_csv(
        os.path.join(out_dir, f"{machine}_best_by_dimension.csv"),
        index=False,
    )

    # -------------------------------------------------------------------------
    # Cross-dimension candidate pool and TOP 10.
    # -------------------------------------------------------------------------
    global_candidates: List[Dict[str, object]] = []
    for dim in range(1, MAX_DIM + 1):
        # D1/D2 can be large; top 80 is more than enough for global top-10.
        pool = sorted(dim_results[dim], key=final_cross_dim_key, reverse=True)
        global_candidates.extend(pool[:TOP_KEEP_PER_DIM])

    # Deduplicate exact subsets.
    dedup: Dict[Tuple[int, ...], Dict[str, object]] = {}
    for r in global_candidates:
        s = tuple(r["subset"])
        if s not in dedup or final_cross_dim_key(r) > final_cross_dim_key(dedup[s]):
            dedup[s] = r

    ranked = sorted(dedup.values(), key=final_cross_dim_key, reverse=True)
    top10 = ranked[:TOP_N_SUBSETS]

    if len(top10) < TOP_N_SUBSETS:
        raise RuntimeError(f"{machine}: only {len(top10)} final candidates")

    top_rows = [result_to_row(machine, r, rank=i + 1) for i, r in enumerate(top10)]
    pd.DataFrame(top_rows).to_csv(
        os.path.join(out_dir, f"{machine}_top10_subsets.csv"),
        index=False,
    )

    # -------------------------------------------------------------------------
    # Feature effectiveness ranking across top candidates + best dimensions.
    # Each candidate contributes 1/dim total mass so larger subsets do not win
    # merely because they contain more features.
    # -------------------------------------------------------------------------
    votes = {j: 0.0 for j in range(N_FEATURES)}
    appearances = {j: 0 for j in range(N_FEATURES)}
    best_dim_appear = {j: 0 for j in range(N_FEATURES)}

    for rank, r in enumerate(top10, start=1):
        rank_weight = (TOP_N_SUBSETS - rank + 1) / TOP_N_SUBSETS
        per_feature = rank_weight / max(int(r["dim"]), 1)
        for j in r["subset"]:
            votes[int(j)] += per_feature
            appearances[int(j)] += 1

    for r in best_by_dim:
        per_feature = 1.0 / max(int(r["dim"]), 1)
        for j in r["subset"]:
            votes[int(j)] += 0.25 * per_feature
            best_dim_appear[int(j)] += 1

    feat_rows = []
    for j in range(N_FEATURES):
        feat_rows.append({
            "machine": machine,
            "feature": BEST_FEATURE_POOL_68[j],
            "effectiveness_vote": float(votes[j]),
            "top10_appearances": int(appearances[j]),
            "best_dimension_appearances": int(best_dim_appear[j]),
        })

    feat_df = pd.DataFrame(feat_rows).sort_values(
        ["effectiveness_vote", "top10_appearances"],
        ascending=[False, False],
    )
    feat_df.insert(0, "feature_rank", np.arange(1, len(feat_df) + 1))
    feat_df.to_csv(
        os.path.join(out_dir, f"{machine}_feature_effectiveness_ranking.csv"),
        index=False,
    )


    elapsed = time.time() - t0
    best = top10[0]
    progress(
        f"{machine} done | evaluated={len(engine.cache)} | "
        f"best_dim={best['dim']} | dim_z={best['dim_z']:.3f}"
    )
    log(f"elapsed_sec={elapsed:.1f}")
    log("TOP 10:")
    if VERBOSE:
        print(pd.DataFrame(top_rows)[[
            "rank", "dim", "dim_z", "normal_only_fitness",
            "target_acceptance", "domain_stability_balance",
            "redundancy_penalty", "features"
        ]].to_string(index=False))

    return {
        "machine": machine,
        "top10": top10,
        "best_by_dim": best_by_dim,
        "feature_ranking": feat_df,
        "evaluated_subset_count": len(engine.cache),
        "elapsed_sec": elapsed,
    }


# =============================================================================
# 8. GLOBAL OUTPUT SUMMARIES
# =============================================================================


# =============================================================================

def save_global_search_summaries(search_results: Dict[str, Dict[str, object]]) -> None:
    top_rows = []
    dim_rows = []
    feat_frames = []

    for machine in MACHINE_TYPES:
        res = search_results[machine]
        for rank, r in enumerate(res["top10"], start=1):
            top_rows.append(result_to_row(machine, r, rank=rank))

        for r in res["best_by_dim"]:
            dim_rows.append(result_to_row(machine, r))

        feat_frames.append(res["feature_ranking"])

    pd.DataFrame(top_rows).to_csv(
        os.path.join(OUTPUT_DIR, "selected_top10_by_machine.csv"),
        index=False,
    )
    pd.DataFrame(dim_rows).to_csv(
        os.path.join(OUTPUT_DIR, "best_by_dimension.csv"),
        index=False,
    )
    pd.concat(feat_frames, ignore_index=True).to_csv(
        os.path.join(OUTPUT_DIR, "feature_effectiveness_ranking.csv"),
        index=False,
    )

    # Machine-independent frequency summary over 8x10 selected subsets.
    global_votes = {f: 0.0 for f in BEST_FEATURE_POOL_68}
    global_apps = {f: 0 for f in BEST_FEATURE_POOL_68}

    for machine in MACHINE_TYPES:
        for rank, r in enumerate(search_results[machine]["top10"], start=1):
            rank_weight = (TOP_N_SUBSETS - rank + 1) / TOP_N_SUBSETS
            per_feature = rank_weight / max(int(r["dim"]), 1)
            for j in r["subset"]:
                f = BEST_FEATURE_POOL_68[int(j)]
                global_votes[f] += per_feature
                global_apps[f] += 1

    rows = [
        {
            "feature": f,
            "global_effectiveness_vote": global_votes[f],
            "appearances_across_8x10": global_apps[f],
        }
        for f in BEST_FEATURE_POOL_68
    ]
    global_feat = pd.DataFrame(rows).sort_values(
        ["global_effectiveness_vote", "appearances_across_8x10"],
        ascending=[False, False],
    ).reset_index(drop=True)
    global_feat.insert(0, "global_rank", np.arange(1, len(global_feat) + 1))
    global_feat.to_csv(
        os.path.join(OUTPUT_DIR, "global_feature_effectiveness_ranking.csv"),
        index=False,
    )


def save_method_config() -> None:
    config = {
        "method": "Frozen68 strict train-only normal-stability subset search",
        "selection_uses_unlabeled_test_features": False,
        "test_files_read_during_selection": False,
        "test_anomaly_labels_used_during_selection": False,
        "test_domain_labels_used_during_selection": False,
        "selection_data": "990 source-normal + 10 target-normal only",
        "candidate_pool_size": 68,
        "max_dimension": MAX_DIM,
        "source_k": SOURCE_K,
        "target_k": TARGET_K,
        "feature_scaler": SCALER_MODE,
        "feature_scaler_fit_data": "pooled 990 source-normal + 10 target-normal train",
        "distance_divide_by_sqrt_dimension": DISTANCE_DIM_NORMALIZE,
        "future_evaluator_score": "min(source_kNN_distance, target_kNN_distance)",
        "objective_weights": {
            "source_compactness": W_SOURCE_COMPACTNESS,
            "target_compactness": W_TARGET_COMPACTNESS,
            "source_tail_stability": W_SOURCE_TAIL_STABILITY,
            "target_tail_stability": W_TARGET_TAIL_STABILITY,
            "target_acceptance": W_TARGET_ACCEPTANCE,
            "domain_stability_balance": W_DOMAIN_STABILITY_BALANCE,
            "redundancy_penalty": W_REDUNDANCY_PENALTY,
        },
        "source_acceptance_quantile": SOURCE_ACCEPTANCE_QUANTILE,
        "source_tail_quantile": SOURCE_TAIL_QUANTILE,
        "target_tail_quantile": TARGET_TAIL_QUANTILE,
        "search": {
            "D1": f"exhaustive {N_FEATURES} singles",
            "D2": f"exhaustive {PAIR_COUNT} pairs",
            "D3_to_D20": "beam forward + random injections",
            "beam_width": BEAM_WIDTH,
            "expansion_pool_size": EXPANSION_POOL_SIZE,
            "random_injections_per_dim": RANDOM_INJECTIONS_PER_DIM,
            "calibration_random_per_dim": CALIBRATION_RANDOM_PER_DIM,
        },
        "dimension_comparison": "robust z relative to same-dimensional baseline",
        "top_n_subsets_reported_per_machine": TOP_N_SUBSETS,
        "fixed68": BEST_FEATURE_POOL_68,
    }
    with open(
        os.path.join(OUTPUT_DIR, "search_method_config.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(config, f, indent=2)


def create_output_zip() -> str:
    base = OUTPUT_DIR.rstrip("/\\")
    zip_path = shutil.make_archive(base, "zip", OUTPUT_DIR)
    return zip_path


def maybe_auto_download(path: str) -> None:
    if not AUTO_DOWNLOAD_ZIP:
        return
    try:
        from google.colab import files as colab_files
        colab_files.download(path)
    except Exception as exc:
        log("Automatic Colab download skipped:", exc)


# =============================================================================
# 9. MAIN -- FEATURE SELECTION / REPORTING ONLY
# =============================================================================

def save_selected_feature_reports(search_results: Dict[str, Dict[str, object]]) -> None:
    top1_rows = []
    long_rows = []
    selected_json: Dict[str, List[Dict[str, object]]] = {}
    top1_json: Dict[str, Dict[str, object]] = {}

    for machine in MACHINE_TYPES:
        selected_json[machine] = []
        for rank, r in enumerate(search_results[machine]["top10"], start=1):
            features = [BEST_FEATURE_POOL_68[int(j)] for j in r["subset"]]
            selected_json[machine].append({
                "rank": rank,
                "dim": int(r["dim"]),
                "dim_z": float(r["dim_z"]),
                "normal_only_fitness": float(r["fitness"]),
                "target_acceptance": float(r["target_acceptance"]),
                "domain_stability_balance": float(r["domain_stability_balance"]),
                "redundancy_penalty": float(r["redundancy_penalty"]),
                "features": features,
            })

        best = search_results[machine]["top10"][0]
        best_features = [BEST_FEATURE_POOL_68[int(j)] for j in best["subset"]]
        top1_rows.append(result_to_row(machine, best, rank=1))
        top1_json[machine] = {
            "dim": int(best["dim"]),
            "dim_z": float(best["dim_z"]),
            "normal_only_fitness": float(best["fitness"]),
            "target_acceptance": float(best["target_acceptance"]),
            "domain_stability_balance": float(best["domain_stability_balance"]),
            "redundancy_penalty": float(best["redundancy_penalty"]),
            "features": best_features,
        }
        for feature_order, feature in enumerate(best_features, start=1):
            long_rows.append({
                "machine": machine,
                "feature_order": feature_order,
                "feature": feature,
            })

    pd.DataFrame(top1_rows).to_csv(
        os.path.join(OUTPUT_DIR, "selected_top1_by_machine.csv"), index=False
    )
    pd.DataFrame(long_rows).to_csv(
        os.path.join(OUTPUT_DIR, "selected_top1_features_long.csv"), index=False
    )
    with open(os.path.join(OUTPUT_DIR, "selected_top10_by_machine.json"), "w", encoding="utf-8") as f:
        json.dump(selected_json, f, indent=2)
    with open(os.path.join(OUTPUT_DIR, "selected_top1_by_machine.json"), "w", encoding="utf-8") as f:
        json.dump(top1_json, f, indent=2)


def build_features_by_machine(search_results: Dict[str, Dict[str, object]]) -> Dict[str, List[str]]:
    return {
        machine: [
            BEST_FEATURE_POOL_68[int(j)]
            for j in search_results[machine]["top10"][0]["subset"]
        ]
        for machine in MACHINE_TYPES
    }


def format_features_by_machine(features_by_machine: Dict[str, List[str]]) -> str:
    lines = ["FEATURES_BY_MACHINE = {"]
    for machine in MACHINE_TYPES:
        lines.append(f'    "{machine}": [')
        for feature in features_by_machine[machine]:
            lines.append(f'        "{feature}",')
        lines.append("    ],")
        lines.append("")
    lines.append("}")
    return "\n".join(lines)


def save_features_by_machine_py(features_by_machine: Dict[str, List[str]]) -> str:
    path = os.path.join(OUTPUT_DIR, "features_by_machine.py")
    with open(path, "w", encoding="utf-8") as f:
        f.write(format_features_by_machine(features_by_machine) + "\n")
    return path


def main() -> None:
    validate_config()
    set_global_seed()
    clean_output_dir()
    save_method_config()

    print("DCASE 2025 Task 2 | Frozen-68 STRICT TRAIN-ONLY feature selection")
    print(f"Input: {INPUT_DIR}")
    print("Selection data: ONLY 990 source-normal + 10 target-normal per machine")
    print("Test recordings are not loaded or used during feature selection.")
    print(f"Search: D=1..{MAX_DIM}, source k={SOURCE_K}, target k={TARGET_K}")

    prepared: Dict[str, PreparedMachine] = {}
    for machine in MACHINE_TYPES:
        prepared[machine] = prepare_machine(machine)

    search_results: Dict[str, Dict[str, object]] = {}
    total_t0 = time.time()
    for machine in MACHINE_TYPES:
        search_results[machine] = search_one_machine(prepared[machine])

    save_global_search_summaries(search_results)
    save_selected_feature_reports(search_results)

    features_by_machine = build_features_by_machine(search_results)
    evaluator_py_path = save_features_by_machine_py(features_by_machine)

    total_elapsed = time.time() - total_t0
    print(f"\nTotal elapsed: {total_elapsed:.1f} s")
    print("Report directory:", OUTPUT_DIR)
    print("Evaluator-ready file:", evaluator_py_path)

    zip_path = create_output_zip()
    print("ZIP:", zip_path)
    maybe_auto_download(zip_path)

    print("\n" + "=" * 100)
    print("FINAL FEATURES_BY_MACHINE -- COPY DIRECTLY INTO EVALUATOR")
    print("=" * 100)
    print(format_features_by_machine(features_by_machine))


if __name__ == "__main__":
    main()
